# 05b — Which genes change with age?

Two views per cell type:
1. **Model weights** from the young-vs-old classifier (needs `figure/{TISSUE}/age_auroc.csv` from 05a).
2. **Trajectories**: change from 1m → 3m (development) vs. 3m → old (aging), from per-mouse averages. Without 1m mice there is no development step.

Genes are filtered first: ambient/blood genes, technical and sex genes, and **other cell types' markers**
(e.g. *Scgb1a1* from club cells showing up in Lung T cells). Findings at the end are Lung's.

### 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from src import aging as ag    # shared functions: edit src/aging.py, not the notebooks

TISSUE = os.environ.get("TISSUE", "Lung")   # set by src/run_tissue.py; edit here to run by hand
P = ag.paths(TISSUE)                        # data/{TISSUE}/..., figure/{TISSUE}/...

adata = sc.read_h5ad(P.processed)

res = pd.read_csv(P.figure("age_auroc.csv"), index_col=0)
ct_means = ag.celltype_means(adata)      # mean expression of every gene in every cell type

### 2. Example cell types

One cell type that passes the batch check and one that fails, shown in detail below. The tissue's reviewed choice if it has one (Lung: fibroblast and CD8 T cell), otherwise the passing type with the highest `cross_age` and the failing type with the highest `logreg`.

In [ ]:
s = ag.settings(TISSUE)
passing, failing = res[res["passes_batch_check"]], res[~res["passes_batch_check"]]
example_pass = s.get("example_pass") or (passing["cross_age"].idxmax() if len(passing) else None)
example_fail = s.get("example_fail") or (failing["logreg"].idxmax() if len(failing) else None)
example_pass, example_fail

### 3. What the filter removes, per cell type

In [ ]:
for ct in [c for c in [example_pass, example_fail] if c]:
    f = ag.foreign_genes(ct_means, ct)
    print(f"{ct}: {len(f)} foreign genes, e.g.", [g for g in ["Scgb1a1", "Sftpc", "Mgp", "Sparc", "Col1a1", "Gzma", "Cd74"] if g in f])

### 4. Top model genes per cell type

Only cell types with mouse-held-out AUROC ≥ 0.8. The `passes_batch_check` column from 05a says which ones to trust.

In [ ]:
good = list(res.index[res["logreg"] >= 0.8])
tops = {ct: ag.top_genes(adata, ct, ag.feature_mask(adata, "own", ct, ct_means), n=20, **ag.main_kw(ct, TISSUE))
        for ct in good}

up_old = pd.DataFrame({ct: t["up_in_old"].values for ct, t in tops.items()})
up_young = pd.DataFrame({ct: t["up_in_young"].values for ct, t in tops.items()})
up_old.to_csv(P.figure("top_genes_up_in_old.csv"), index=False)
up_young.to_csv(P.figure("top_genes_up_in_young.csv"), index=False)
print(res.loc[good, ["logreg", "cross_age", "passes_batch_check"]].round(2))
up_old

### 5. Genes shared across cell types, with batch flags

In [ ]:
def tag(g):
    if g in ag.CIRCADIAN:
        return "circadian (time of collection)"
    if g in ag.DISSOCIATION:
        return "dissociation stress"
    return ""

for name, table in [("up in old", up_old), ("up in young", up_young)]:
    counts = table.melt(value_name="gene")["gene"].value_counts()
    shared = counts[counts >= 2].to_frame("n_cell_types")
    shared["flag"] = [tag(g) for g in shared.index]
    print(f"\n{name}, in 2+ cell types:")
    print(shared)

### 6. Development vs. aging, the passing example

In [ ]:
ct = example_pass
tr = ag.label_patterns(ag.trajectories(adata, ct, exclude=ag.foreign_genes(ct_means, ct), **ag.main_kw(ct, TISSUE)))
print(tr["pattern"].value_counts())
for p, genes in ag.pattern_top(tr).items():
    print(f"\n{p}:", genes)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(data=tr, x="development", y="aging", hue="pattern", s=12, ax=ax, linewidth=0)
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("1m → 3m change (log)"); ax.set_ylabel("3m → old change (log)")
ax.set_title(ct)
fig.savefig(P.figure(f"dev_vs_aging_{ag.slug(ct)}.png"), dpi=200, bbox_inches="tight")

### 7. Aging-only genes for every cell type that passed the batch check

In [ ]:
passed = list(res.index[res["passes_batch_check"]])
rows = []
for ct in passed:
    t = ag.label_patterns(ag.trajectories(adata, ct, exclude=ag.foreign_genes(ct_means, ct), **ag.main_kw(ct, TISSUE)))
    sel = t[t.pattern == "aging only"].sort_values("aging", key=abs, ascending=False)
    rows += [{"cell_type": ct, "gene": g, "aging_change": round(v, 2), "flag": tag(g)}
             for g, v in sel["aging"].head(15).items()]
aging_genes = pd.DataFrame(rows)
aging_genes.to_csv(P.figure("aging_only_genes.csv"), index=False)
aging_genes

### 8. Dissociation and circadian check for one cell type

If these genes show large aging changes, treat stress-like genes in that cell type with caution.

In [ ]:
flag = tr.loc[tr.index.isin(ag.DISSOCIATION + ag.CIRCADIAN), ["development", "aging", "pattern"]].round(2)
flag.sort_values("aging", ascending=False)

### 9. The failing example: development vs. aging

This cell type fails the batch check, so read it as a look at subsets within the cell type, not as a shared aging program. In Lung (CD8 T cells), old mice have more memory-like and fewer naive cells, so "aging only" genes there may reflect that shift.

In [ ]:
ct = example_fail
tr_fail = ag.label_patterns(ag.trajectories(adata, ct, exclude=ag.foreign_genes(ct_means, ct), **ag.main_kw(ct, TISSUE)))
print(tr_fail["pattern"].value_counts())
for p, genes in ag.pattern_top(tr_fail).items():
    print(f"\n{p}:", genes)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(data=tr_fail, x="development", y="aging", hue="pattern", s=12, ax=ax, linewidth=0)
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("1m → 3m change (log)"); ax.set_ylabel("3m → old change (log)")
ax.set_title(ct)
fig.savefig(P.figure(f"dev_vs_aging_{ag.slug(ct)}.png"), dpi=200, bbox_inches="tight")

In [ ]:
flag = tr_fail.loc[tr_fail.index.isin(ag.DISSOCIATION + ag.CIRCADIAN), ["development", "aging", "pattern"]].round(2)
flag.sort_values("aging", ascending=False)

### Findings (Lung)

- **The "own" filter works only partly.** Collagens and *Mgp* are gone from T cells and monocytes, but it compares log-scale values, so abundant ambient RNA can pass. *Scgb1a1* passed in 6 cell types, so it is now removed by name through `ag.AMBIENT`. *Mgp* remains in alveolar macrophages, and *Sftpc* (type II pneumocyte surfactant) remains in neutrophils and bronchial smooth muscle. Treat those hits as ambient.
- **Batch signs remain:** *Dbp* (circadian) is "up in young" in several cell types, and *Atf3* and *Ier3* (dissociation) are "up in old" in 2.
- **Fibroblasts, which pass the batch check:** *Cdkn1a* (p21, a senescence marker) and complement genes (*C3*, *Cfb*) are up in old, and collagens down. *Cdkn1a* is also "aging only" in the trajectory view. This is the best lead for Phase 6.
- **Inflammation genes** (*Lrg1*, *Slpi*, *Hp*, *Il1b*, *Lcn2*) recur across cell types, mostly ones that fail the batch check, so they are suggestive only.
- **CD8 T cells:** "aging only" genes (*S100a4*, *S100a6*, *Lgals1*, *Nkg7*, *Gzmm*) fit a shift toward memory-like cells. *Sftpc* under "reverses" is contamination.
- **Neutrophil rows** in `lung_aging_only_genes.csv` should be ignored: that cell type passes the batch check, but its shuffled control is too high.